# Content Moderation — BigQuery AI Functions

An end-to-end content moderation pipeline that composes five AI functions:

1. **Generate** sample social media posts with `AI.GENERATE_TABLE`
2. **Flag** posts needing review with `AI.IF`
3. **Classify** flagged posts by violation type with `AI.CLASSIFY`
4. **Score** severity of flagged posts with `AI.SCORE`
5. **Summarize** moderation results with `AI.GENERATE`

**What this demonstrates:**
- Using `AI.IF` as a binary filter step — efficiently gate expensive downstream processing
- Composing filter → classify → score → summarize in one pipeline
- Each function adds value: `AI.IF` reduces volume, `AI.CLASSIFY` categorizes, `AI.SCORE` prioritizes

**Functions used:** [`AI.GENERATE_TABLE`](../../functions/ai_generate_table/) | [`AI.IF`](../../functions/ai_if/) | [`AI.CLASSIFY`](../../functions/ai_classify/) | [`AI.SCORE`](../../functions/ai_score/) | [`AI.GENERATE`](../../functions/ai_generate/)

**Prerequisites:** [Setup guide](../../setup/) | [Function reference](../../RESOURCES.md)

---
## Setup

Set your project and location, authenticate, and create shared resources.

> This workflow uses `AI.GENERATE_TABLE` (requires a connection and remote model) and `AI.IF`, `AI.CLASSIFY`, `AI.SCORE`, `AI.GENERATE` (no model needed). See the [Setup Reference](../../setup/) for details.

In [1]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks
CONNECTION_ID = 'bq_ai_functions'  # Shared connection

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [2]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install('google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm')

In [3]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [4]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready


In [5]:
import subprocess as _sp, json as _json

# Create connection (idempotent)
_sp.run(['bq', 'mk', '--connection', '--location', LOCATION,
         '--connection_type', 'CLOUD_RESOURCE',
         '--project_id', PROJECT_ID, CONNECTION_ID],
        capture_output=True, text=True)

# Get service account and grant Vertex AI User role
r = _sp.run(['bq', 'show', '--connection', '--format=json',
             '--project_id', PROJECT_ID, '--location', LOCATION, CONNECTION_ID],
            capture_output=True, text=True, check=True)
sa = _json.loads(r.stdout)['cloudResource']['serviceAccountId']
_sp.run(['gcloud', 'projects', 'add-iam-policy-binding', PROJECT_ID,
         f'--member=serviceAccount:{sa}', '--role=roles/aiplatform.user', '--quiet'],
        capture_output=True, text=True)
print(f'Connection {CONNECTION_ID} ready (SA: {sa})')

Connection bq_ai_functions ready (SA: bqcx-1026793852137-hx0g@gcp-sa-bigquery-condel.iam.gserviceaccount.com)


In [6]:
# Create remote Gemini model for AI.GENERATE_TABLE (idempotent)
client.query(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`
  REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
  OPTIONS (endpoint = \'gemini-2.5-flash\')
''').result()
print('Model gemini_flash ready')

Model gemini_flash ready


---
## Step 1 — Generate sample posts with AI.GENERATE_TABLE

Generate 20 social media posts from seed topics — a realistic mix of clean content and problematic posts. Each seed row describes a post category and topic; `AI.GENERATE_TABLE` generates a username and post text for each one.

In [7]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_mod_posts` AS
SELECT post_id, username, post_text
FROM AI.GENERATE_TABLE(
  MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`,
  (SELECT
    post_id,
    CONCAT(
      'Generate one realistic social media post. Category: ', category,
      '. Topic: ', topic,
      '. Write 1-2 sentences. Create a believable username.'
    ) AS prompt
   FROM UNNEST([
     STRUCT(1 AS post_id, 'clean' AS category, 'positive product review of electronics' AS topic),
     STRUCT(2, 'clean', 'travel experience in another country'),
     STRUCT(3, 'clean', 'cooking recipe tip'),
     STRUCT(4, 'clean', 'tech discussion about a new phone'),
     STRUCT(5, 'clean', 'fitness motivation'),
     STRUCT(6, 'clean', 'book recommendation'),
     STRUCT(7, 'clean', 'pet appreciation post'),
     STRUCT(8, 'clean', 'hobby sharing about painting'),
     STRUCT(9, 'clean', 'local restaurant review'),
     STRUCT(10, 'clean', 'gardening update'),
     STRUCT(11, 'clean', 'music discovery'),
     STRUCT(12, 'clean', 'weekend hiking recap'),
     STRUCT(13, 'problematic', 'spam with fake URLs and too-good-to-be-true offers'),
     STRUCT(14, 'problematic', 'mild harassment targeting appearance'),
     STRUCT(15, 'problematic', 'obvious health misinformation about vaccines'),
     STRUCT(16, 'problematic', 'engagement bait clickbait'),
     STRUCT(17, 'problematic', 'spam promoting a get-rich-quick scheme'),
     STRUCT(18, 'problematic', 'personal attack and insult'),
     STRUCT(19, 'problematic', 'conspiracy theory misinformation'),
     STRUCT(20, 'problematic', 'phishing attempt disguised as urgent account warning')
   ])),
  STRUCT('username STRING, post_text STRING' AS output_schema)
)
'''
client.query(query).result()

posts = client.query(f'''
  SELECT post_id, username, post_text
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_posts`
  ORDER BY post_id
''').to_dataframe()
print(f'Generated {len(posts)} posts')
posts.head(10)

Generated 20 posts


,post_id,username,post_text
0,1,PixelPerfectPro,Just received my new UltraView monitor and the picture quality is absolutely stunning! My games and movies have never looked so vibrant and clear.
1,2,WanderlustWendy,Just spent the most incredible week exploring the ancient ruins and vibrant markets of Rome! Every corner was a postcard. Can't wait to share more photos!
2,3,Culinary_Corner,Here's a tip that elevates any dish: always bloom your spices in a little oil before adding other ingredients. It really brings out their full flavor!
3,4,AndroidAdept,Just pre-ordered the new Galaxy S25 Ultra! I'm really hyped about the rumored battery life improvements and the upgraded S-Pen features this time around.
4,5,FitLifeJourney,"Feeling stronger every day! Consistency is key, keep pushing towards your goals."
5,6,BookwormBeth,"Just finished 'Project Hail Mary' by Andy Weir and it was absolutely brilliant! Such a fun, clever, and heartwarming read, I highly recommend it!"
6,7,PawsitivelyHappy,My furry best friend always knows how to brighten my day! So grateful for all the snuggles and wagging tails. #petlove #dogmom
7,8,ArtisticSoul,"Just finished my latest watercolor piece! It's so relaxing to get lost in the colors and brushstrokes, definitely my happy place."
8,9,FoodieFiesta88,"Just tried the new Italian place downtown, 'Pasta Perfection'! Their lasagna was absolutely divine, and the service was super friendly. Highly recommend checking it out!"
9,10,PlantMama,The hydrangeas are blooming beautifully this year! Loving all the vibrant colors in the garden.


---
## Step 2 — Flag posts with AI.IF

`AI.IF` evaluates a natural language condition and returns TRUE or FALSE. Use it as a binary filter to identify posts needing review — this gates the more expensive downstream classification and scoring steps.

In [8]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_mod_flagged` AS
SELECT
  post_id,
  username,
  post_text,
  AI.IF(
    CONCAT(
      'This social media post contains problematic content such as spam, ',
      'harassment, hate speech, or misinformation: ',
      post_text
    )
  ) AS needs_review
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_posts`
'''
client.query(query).result()

flagged = client.query(f'''
  SELECT needs_review, COUNT(*) AS count
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_flagged`
  GROUP BY needs_review
''').to_dataframe()
print('AI.IF flagging results:')
for _, row in flagged.iterrows():
    label = 'Flagged for review' if row['needs_review'] else 'Clean'
    print(f'  {label}: {row["count"]} posts')

AI.IF flagging results:
  Clean: 12 posts
  Flagged for review: 8 posts


---
## Step 3 — Classify flagged posts with AI.CLASSIFY

`AI.CLASSIFY` categorizes each flagged post into a violation type. Only flagged posts are classified — `AI.IF` already filtered out the clean ones, saving API calls.

In [9]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_mod_classified` AS
SELECT
  post_id,
  username,
  post_text,
  AI.CLASSIFY(
    post_text,
    ['harassment', 'spam', 'hate_speech', 'misinformation']
  ) AS violation_type
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_flagged`
WHERE needs_review = TRUE
'''
client.query(query).result()

classified = client.query(f'''
  SELECT violation_type, COUNT(*) AS count
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_classified`
  GROUP BY violation_type
  ORDER BY count DESC
''').to_dataframe()
print('Violation type distribution:')
for _, row in classified.iterrows():
    print(f'  {row["violation_type"]}: {row["count"]}')

Violation type distribution:
  misinformation: 3
  spam: 3
  harassment: 2


---
## Step 4 — Score severity with AI.SCORE

`AI.SCORE` rates each flagged post on a 0–1 severity scale. Higher scores indicate more severe violations that need immediate attention.

In [10]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_mod_scored` AS
SELECT
  c.post_id,
  c.username,
  c.post_text,
  c.violation_type,
  AI.SCORE(
    CONCAT(
      'Rate the severity of this content violation. ',
      'Score 0 for borderline/mild issues, 1 for severe violations ',
      'requiring immediate action: ',
      c.post_text
    )
  ) AS severity_score
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_classified` c
'''
client.query(query).result()

scored = client.query(f'''
  SELECT post_id, username, violation_type,
    ROUND(severity_score, 2) AS severity_score, post_text
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_scored`
  ORDER BY severity_score DESC
''').to_dataframe()
print('Flagged posts ranked by severity:')
scored

Flagged posts ranked by severity:


,post_id,username,violation_type,severity_score,post_text
0,19,TruthSeeker777,misinformation,1.0,"They're spraying us from above again, making everyone sick. Wake up, people, it's not just clouds, it's population control."
1,15,WakeUpSheeple77,misinformation,1.0,"Did you know that vaccine manufacturers are hiding the fact that these shots can cause infertility and alter your DNA? Do your own research, don't trust the mainstream media!"
2,17,WealthWizard88,spam,1.0,"Tired of the 9-5 grind? My new system guarantees you'll make $10,000 this week working from home! DM me ""INFO"" to start your financial freedom journey NOW!"
3,20,AccountSafeguard,spam,1.0,Urgent: We've detected suspicious login attempts on your account. Click this link immediately to secure your profile and prevent permanent suspension: bit.ly/account-alert
4,13,TravelDealsNow,spam,1.0,"🚨 AMAZING OFFER! Click here to claim your FREE vacation to Hawaii plus $5,000 cash! Limited time only, act fast: tinyurl.com/hawaiidream24"
5,18,KeyboardWarrior99,harassment,0.0,"Honestly, your opinion is as worthless as your fashion sense. Did you even think before you typed that garbage?"
6,14,StylePolice_247,harassment,0.0,Did you even look in a mirror before you left the house today? That outfit is... certainly a choice.
7,16,RealTalk_Tina,misinformation,0.0,I just saw something that completely changed my perspective on [insert vaguely controversial topic here]... You won't BELIEVE what they're hiding! Hit 'Like' if you think the truth needs to come out! 🤫


### Step 4b — Action items

Show the highest-severity posts that need immediate attention (severity > 0.7).

In [11]:
query = f'''
SELECT post_id, username, violation_type,
  ROUND(severity_score, 2) AS severity_score, post_text
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_scored`
WHERE severity_score > 0.7
ORDER BY severity_score DESC
'''
urgent = client.query(query).to_dataframe()
print(f'{len(urgent)} posts need immediate attention:')
urgent

5 posts need immediate attention:


,post_id,username,violation_type,severity_score,post_text
0,19,TruthSeeker777,misinformation,1.0,"They're spraying us from above again, making everyone sick. Wake up, people, it's not just clouds, it's population control."
1,15,WakeUpSheeple77,misinformation,1.0,"Did you know that vaccine manufacturers are hiding the fact that these shots can cause infertility and alter your DNA? Do your own research, don't trust the mainstream media!"
2,17,WealthWizard88,spam,1.0,"Tired of the 9-5 grind? My new system guarantees you'll make $10,000 this week working from home! DM me ""INFO"" to start your financial freedom journey NOW!"
3,20,AccountSafeguard,spam,1.0,Urgent: We've detected suspicious login attempts on your account. Click this link immediately to secure your profile and prevent permanent suspension: bit.ly/account-alert
4,13,TravelDealsNow,spam,1.0,"🚨 AMAZING OFFER! Click here to claim your FREE vacation to Hawaii plus $5,000 cash! Limited time only, act fast: tinyurl.com/hawaiidream24"


---
## Step 5 — Executive summary with AI.GENERATE

Aggregate all pipeline results and generate an executive summary of the moderation run.

In [12]:
query = f'''
WITH stats AS (
  SELECT
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_posts`) AS total_posts,
    (SELECT COUNTIF(needs_review) FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_flagged`) AS flagged_count,
    (SELECT COUNTIF(NOT needs_review) FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_flagged`) AS clean_count,
    (SELECT STRING_AGG(CONCAT(violation_type, ': ', CAST(cnt AS STRING)), ', ')
     FROM (SELECT violation_type, COUNT(*) AS cnt
           FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_classified`
           GROUP BY violation_type)) AS violation_breakdown,
    (SELECT ROUND(AVG(severity_score), 2) FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_scored`) AS avg_severity,
    (SELECT COUNTIF(severity_score > 0.7) FROM `{PROJECT_ID}.{DATASET_ID}.workflow_mod_scored`) AS urgent_count
)
SELECT (AI.GENERATE(
  CONCAT(
    'Write a brief executive summary (2-3 paragraphs) of this content moderation pipeline run. ',
    'Total posts: ', CAST(total_posts AS STRING),
    '. Flagged for review: ', CAST(flagged_count AS STRING),
    '. Clean: ', CAST(clean_count AS STRING),
    '. Violation breakdown: ', IFNULL(violation_breakdown, 'none'),
    '. Average severity: ', CAST(IFNULL(avg_severity, 0) AS STRING),
    '. Posts needing immediate action (severity > 0.7): ', CAST(IFNULL(urgent_count, 0) AS STRING),
    '. End with a one-sentence takeaway.'
  )
)).result AS executive_summary
FROM stats
'''
df = client.query(query).to_dataframe()
print(df.iloc[0]['executive_summary'])

A recent content moderation pipeline run processed 20 posts, identifying 8 (40%) requiring further review due to policy violations, with the remaining 12 posts deemed clean. The flagged content consisted of 2 instances of harassment, 3 of misinformation, and 3 of spam, indicating a diverse range of policy infringements.

The overall severity of the flagged content averaged 0.63, suggesting a moderate level of concern across the violations. Critically, 5 of the 8 flagged posts were identified with a severity exceeding 0.7, demanding immediate attention and intervention to mitigate potential harm.

This run highlights a significant proportion of high-severity content requiring swift moderation action.


---
## Cleanup

Drop tables created by this workflow. Shared resources (dataset, connection, models) are left for other notebooks.

In [13]:
for table in ['workflow_mod_posts', 'workflow_mod_flagged', 'workflow_mod_classified', 'workflow_mod_scored']:
    client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.{table}', not_found_ok=True)
    print(f'Dropped {table}')

Dropped workflow_mod_posts
Dropped workflow_mod_flagged
Dropped workflow_mod_classified
Dropped workflow_mod_scored


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [14]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')